In [34]:
import pandas as pd
import numpy as np
from prophet import Prophet
from tqdm import tqdm
from functools import reduce
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import VotingRegressor

In [35]:
df=pd.read_csv("../ISI_dataset\merged_potato_reservoir.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\shrey\AppData\Local\Temp\ipykernel_16128\2810355396.py:1: SyntaxWarning: invalid escape sequence '\m'
  df=pd.read_csv("../ISI_dataset\merged_potato_reservoir.csv")


,state_name,crop_name,apy_item_interval_start,temperature_recorded_date,state_temperature_max_val,state_temperature_min_val,state_rainfall_val,yield,FRL,Live Cap FRL,Level,Current Live Storage
0,Andhra Pradesh,potato,2012,2012-01-01,31.05,17.86,10.92,15.31692,337.152584,0.198465,174.568780,0.782629
1,Andhra Pradesh,potato,2012,2012-01-02,32.33,17.19,2.29,15.31692,337.152584,0.198465,174.565447,0.781853
2,Andhra Pradesh,potato,2012,2012-01-03,33.15,16.89,0.71,15.31692,337.152584,0.198465,174.563225,0.781850
3,Andhra Pradesh,potato,2012,2012-01-04,33.95,17.70,0.00,15.31692,337.152584,0.198465,174.557669,0.780744
4,Andhra Pradesh,potato,2012,2012-01-05,33.79,17.61,0.00,15.31692,337.152584,0.198465,174.548780,0.780603


In [36]:
df['temperature_recorded_date'] = pd.to_datetime(df['temperature_recorded_date'])
df['year'] = df['temperature_recorded_date'].dt.year

In [37]:
# Use only data till 2022 for training
df = df[df['year'] < 2023].copy()

# Drop unreliable states
df = df[~df['state_name'].isin(['Andhra Pradesh', 'Tamil Nadu', 'Telangana','West Bengal'])]

# Group annually to match 2023 structure
df_annual = df.groupby(['state_name', 'crop_name', 'year']).agg({
    'state_rainfall_val': 'sum',
    'state_temperature_max_val': 'mean',
    'state_temperature_min_val': 'mean',
    'Live Cap FRL': 'mean',
    'FRL': 'mean',
    'Level': 'mean',
    'Current Live Storage': 'mean',
    'yield': 'mean'
}).reset_index()

In [38]:
df_annual.head()

,state_name,crop_name,year,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,yield
0,Chhattisgarh,potato,2006,1364.20,33.776658,18.041726,1.365667,377.82,349.127932,1.185512,3.97509
1,Chhattisgarh,potato,2007,1349.22,33.980082,18.029699,1.365667,377.82,347.372466,0.956266,4.21392
2,Chhattisgarh,potato,2008,1242.55,33.966721,18.241612,1.365667,377.82,361.029144,0.794941,4.21896
3,Chhattisgarh,potato,2011,1293.99,34.128356,17.759836,1.365667,377.82,372.145735,0.792748,5.36698
4,Chhattisgarh,potato,2012,1394.81,34.314153,18.043443,1.365667,377.82,373.008725,0.926530,6.88596


In [39]:
# One-hot encode 'state_name'
df_encoded = pd.get_dummies(df_annual, columns=['state_name'])

# Define features: original + one-hot encoded state columns
state_columns = [col for col in df_encoded.columns if col.startswith('state_name_')]

In [40]:
# Define features and target
features = ['state_rainfall_val', 'state_temperature_max_val', 'state_temperature_min_val', 'Live Cap FRL', 'FRL','Level','Current Live Storage']+ state_columns

# Split manually by year
train_df = df_encoded[df_encoded['year'] <= 2020]
test_df = df_encoded[df_encoded['year'].between(2021, 2022)]

In [41]:
X_train = train_df[features]
y_train = train_df['yield']
X_test = test_df[features]
y_test = test_df['yield']

In [42]:
# Models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR()
}

# Results container
results = []

# Loop through models
for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        'Model': name,
        'Train R²': round(train_r2, 4),
        'Test R²': round(test_r2, 4),
        'Train RMSE': round(train_rmse, 2),
        'Test RMSE': round(test_rmse, 2)
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Test R²', ascending=False))

                      Model  Train R²  Test R²  Train RMSE  Test RMSE
0         Linear Regression    0.9281   0.6524        1.89       6.20
1             Random Forest    0.9853   0.5650        0.86       6.93
3         Gradient Boosting    0.9971   0.5056        0.38       7.39
2                   XGBoost    1.0000   0.4681        0.00       7.66
4  Support Vector Regressor    0.3142  -0.0787        5.85      10.91


In [43]:
# Initialize individual models
lr = LinearRegression()
rf = RandomForestRegressor()
gb = GradientBoostingRegressor(random_state=42)

# Ensemble model
ensemble = VotingRegressor(estimators=[
    ('lr', lr),
    ('rf', rf),
    ('gb', gb)
])

# Fit ensemble
ensemble.fit(X_train, y_train)

# Predict
train_pred = ensemble.predict(X_train)
test_pred = ensemble.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("📊 Ensemble Performance:")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")


📊 Ensemble Performance:
Train R²: 0.9816, Test R²: 0.5787
Train RMSE: 0.9584, Test RMSE: 6.8203


In [44]:
# --- 1. Define function to forecast any single feature using Prophet ---
def forecast_feature_prophet(df, feature_name):
    forecast_data = []

    for (state, crop), group in tqdm(df.groupby(['state_name', 'crop_name'])):
        yearly_data = group.groupby('year')[feature_name].mean().reset_index()

        if yearly_data.shape[0] < 4:
            continue

        prophet_df = yearly_data.rename(columns={'year': 'ds', feature_name: 'y'})
        prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

        try:
            model = Prophet()
            model.fit(prophet_df)

            future = pd.DataFrame({'ds': [pd.to_datetime('2023')]})
            forecast = model.predict(future)
            yhat = forecast['yhat'].values[0]

            forecast_data.append({
                'state_name': state,
                'crop_name': crop,
                feature_name: yhat
            })
        except:
            continue

    return pd.DataFrame(forecast_data)

# --- 2. Forecast each feature separately ---
df_rain = forecast_feature_prophet(df, 'state_rainfall_val')
df_temp_max = forecast_feature_prophet(df, 'state_temperature_max_val')
df_temp_min = forecast_feature_prophet(df, 'state_temperature_min_val')
df_livecap = forecast_feature_prophet(df, 'Live Cap FRL')
df_frl = forecast_feature_prophet(df, 'FRL')
df_level = forecast_feature_prophet(df, 'Level')
df_cls = forecast_feature_prophet(df, 'Current Live Storage')

# --- 3. Merge all forecasted dataframes ---
from functools import reduce
dfs = [df_rain, df_temp_max, df_temp_min, df_livecap, df_frl, df_level, df_cls]
df_2023 = reduce(lambda left, right: pd.merge(left, right, on=['state_name', 'crop_name'], how='outer'), dfs)

# --- 4. One-hot encode state_name ---
df_2023_encoded = df_2023.copy()  # Keep original columns
state_names = df_2023_encoded[['state_name', 'crop_name']]  # Keep for merging later

df_2023_encoded = pd.get_dummies(df_2023_encoded, columns=['state_name'])
df_2023_encoded = pd.concat([state_names, df_2023_encoded.drop(columns=['crop_name'])], axis=1)


  0%|          | 0/5 [00:00<?, ?it/s]22:18:02 - cmdstanpy - INFO - Chain [1] start processing
22:18:02 - cmdstanpy - INFO - Chain [1] done processing
 20%|██        | 1/5 [00:00<00:01,  3.91it/s]22:18:03 - cmdstanpy - INFO - Chain [1] start processing
22:18:03 - cmdstanpy - INFO - Chain [1] done processing
 40%|████      | 2/5 [00:00<00:00,  3.53it/s]22:18:03 - cmdstanpy - INFO - Chain [1] start processing
22:18:03 - cmdstanpy - INFO - Chain [1] done processing
 60%|██████    | 3/5 [00:00<00:00,  3.60it/s]22:18:03 - cmdstanpy - INFO - Chain [1] start processing
22:18:03 - cmdstanpy - INFO - Chain [1] done processing
 80%|████████  | 4/5 [00:01<00:00,  3.50it/s]22:18:03 - cmdstanpy - INFO - Chain [1] start processing
22:18:04 - cmdstanpy - INFO - Chain [1] done processing
  0%|          | 0/5 [00:00<?, ?it/s]22:18:04 - cmdstanpy - INFO - Chain [1] start processing
22:18:04 - cmdstanpy - INFO - Chain [1] done processing
 20%|██        | 1/5 [00:00<00:01,  3.00it/s]22:18:04 - cmdstanpy - 

In [45]:
df_2023_encoded.head()

,state_name,crop_name,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,state_name_Chhattisgarh,state_name_Jharkhand,state_name_Karnataka,state_name_Uttar Pradesh,state_name_Uttarakhand
0,Chhattisgarh,potato,3.722373,33.764963,18.273648,1.365667,377.820000,353.271640,1.530672,True,False,False,False,False
1,Jharkhand,potato,3.345112,33.109076,18.655014,0.404750,296.267500,292.152319,0.213347,False,True,False,False,False
2,Karnataka,potato,3.332718,33.710382,17.459753,1.539062,597.731875,594.134793,0.790176,False,False,True,False,False
3,Uttar Pradesh,potato,2.342267,34.031371,17.007908,2.898000,183.212432,162.407597,1.229087,False,False,False,True,False
4,Uttarakhand,potato,4.037045,29.747857,14.455743,1.713670,461.933333,444.391264,0.947704,False,False,False,False,True


In [46]:
X_2023 = df_2023_encoded[features]

# Use your trained ensemble model
y_2023_pred = ensemble.predict(X_2023)

# Add prediction to the dataframe
df_2023_encoded['predicted_yield'] = y_2023_pred

# Select output
output_2023 = df_2023_encoded[['crop_name'] + [col for col in df_2023_encoded.columns if col.startswith('state_name_')] + ['predicted_yield']]


In [47]:
# Convert dummy columns back to state_name
state_names = df_2023_encoded[[col for col in df_2023_encoded.columns if col.startswith('state_name_')]].idxmax(axis=1)
state_names = state_names.str.replace('state_name_', '')

# Final output
final_2023_yield = pd.DataFrame({
    'state_name': state_names,
    'crop_name': df_2023_encoded['crop_name'],
    'predicted_yield_2023': df_2023_encoded['predicted_yield']
})

print(final_2023_yield)
final_2023_yield.to_csv("../yield_prediction.csv", index=False)

      state_name crop_name  predicted_yield_2023
0   Chhattisgarh    potato              5.073596
1      Jharkhand    potato              6.070352
2      Karnataka    potato             15.045357
3  Uttar Pradesh    potato             24.192312
4    Uttarakhand    potato             10.976483


In [48]:
# Step 1: Get actual yields from 2019 to 2022
df_recent = df_annual[df_annual['year'].between(2019, 2022)].copy()

# Pivot to get each year's yield as a column
yield_table = df_recent.pivot_table(
    index=['state_name', 'crop_name'],
    columns='year',
    values='yield'
).reset_index()

# Rename columns for clarity
yield_table = yield_table.rename(columns={
    2019: 'yield_2019',
    2020: 'yield_2020',
    2021: 'yield_2021',
    2022: 'yield_2022'
})

# Step 2: Prepare 2023 predicted yield
df_2023_yield = df_2023_encoded[['state_name', 'crop_name', 'predicted_yield']].copy()
df_2023_yield = df_2023_yield.rename(columns={'predicted_yield': 'yield_2023'})

# Step 3: Merge the 2023 predicted yield into the table
final_yield_table = pd.merge(yield_table, df_2023_yield, on=['state_name', 'crop_name'], how='left')

# Display final table
print(final_yield_table)


      state_name crop_name  yield_2019  yield_2020  yield_2021  yield_2022  \
0   Chhattisgarh    potato     5.79702     7.14202     6.95692     7.21700   
1      Jharkhand    potato         NaN     7.30545         NaN         NaN   
2      Karnataka    potato    15.48805    18.78620    14.54044    17.81122   
3  Uttar Pradesh    potato    26.91971    29.58990    35.00000    35.00000   
4    Uttarakhand    potato    11.85946    11.55998    12.93403    12.52321   

   yield_2023  
0    5.073596  
1    6.070352  
2   15.045357  
3   24.192312  
4   10.976483  
